# steering-mechanics — zero-setup demo (Colab, free GPU)

Runs a [brainscope](https://github.com/moudrkat/brainscope) server on Colab's
GPU, then dissects a steering vector with this repo's tools — dose-response,
component attribution, the head tug-of-war. No local setup, no private data.

**Runtime → Change runtime type → GPU** first.


## 1. Install


In [ ]:
!pip -q install brainscope steering-mechanics 2>/dev/null || (
  git clone -q https://github.com/moudrkat/brainscope &&
  git clone -q https://github.com/moudrkat/steering-mechanics &&
  pip -q install -e brainscope -e steering-mechanics )


## 2. Start brainscope on the GPU
A small public model + a J-lens fit, served in the background.


In [ ]:
import subprocess, time, urllib.request, os
srv = subprocess.Popen(['brainscope','--model','Qwen/Qwen2.5-0.5B-Instruct',
                        '--port','8010','--no-browser','--device','cuda'])
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:8010/info', timeout=3); break
    except Exception: time.sleep(5)
os.environ['BRAINSCOPE_BASE']='http://localhost:8010'
print('brainscope up')


## 3. Make a public vector from a word
brainscope mints a J-lens direction for any word — no private data needed.


In [ ]:
import urllib.request, json
def post(path, body):
    r=urllib.request.Request('http://localhost:8010'+path, json.dumps(body).encode(),
                             {'Content-Type':'application/json'})
    return json.loads(urllib.request.urlopen(r, timeout=120).read())
post('/jlens', {'on': True})
name = post('/jlens/direction', {'text':'reminder'})['name']
print('made direction:', name)


## 4. Discover its intent, then dissect it


In [ ]:
!steermech-discover --key demo --id {name} --layer 6
!python steering-mechanics/experiments/dose_response.py --id {name} --layer 6 --scales 1 3 6


## 5. Render the figures


In [ ]:
!cd steering-mechanics && python -m steermech.plot
from IPython.display import Image; Image('steering-mechanics/fig/tug_of_war.gif')


---
Full research + findings: https://github.com/moudrkat/steering-mechanics
